# 004 · Exact Fractions

> **Standard Library › Numbers & Math** · Python 3.14

`fractions.Fraction` represents a **rational number** — a numerator over a denominator — exactly. `1/3 + 1/6` is precisely `1/2`, with no rounding at all. Because Python integers are unbounded, fractions never overflow either.

**In this notebook you will learn:**

- creating fractions from integers, strings, floats and decimals
- exact arithmetic and how fractions mix with `int`, `float` and `Decimal`
- recovering "nice" fractions from floats with `limit_denominator`
- `as_integer_ratio`, converting back, and formatting
- when to choose `Fraction` vs `Decimal` vs `float`

## .NET mental model

There is **no rational number type in the .NET BCL**. The nearest building block is `System.Numerics.BigInteger` — you would store a numerator/denominator pair and write the normalisation (GCD reduction), arithmetic and comparison operators yourself, or pull in a community NuGet package.

| Need | .NET | Python |
|---|---|---|
| Exact 1/3 | hand-written `struct Rational(BigInteger Num, BigInteger Den)` | `Fraction(1, 3)` |
| Reduce to lowest terms | `BigInteger.GreatestCommonDivisor` by hand | automatic |
| Arbitrary size | `BigInteger` | built in (Python `int`) |
| Base-10 exactness only | `decimal` | `Decimal` *(see Standard Library › Numbers & Math › 003 · Decimal Arithmetic)* |

## Creating fractions

`Fraction(numerator, denominator)` is always stored in **lowest terms** with a positive denominator.

In [ ]:
from fractions import Fraction

print(Fraction(3, 4))      # 3/4
print(Fraction(6, 8))      # 3/4 — reduced automatically
print(Fraction(3, -4))     # -3/4 — sign moves to the numerator
print(Fraction(5))         # 5   — denominator 1

Strings are the most convenient source: both `"n/d"` and decimal notation are accepted.

In [ ]:
print(Fraction("3/4"), Fraction("0.75"), Fraction("-1.5e-2"), Fraction("  7/21 "))

A zero denominator raises `ZeroDivisionError`:

In [ ]:
try:
    Fraction(1, 0)
except ZeroDivisionError as e:
    print(f"{type(e).__name__}: {e}")

## From floats and decimals

⚠️ **Gotcha:** like `Decimal`, a `Fraction` built from a float captures the float's **exact** binary value — which for `0.1` is a fraction with a power-of-two denominator, not `1/10`. Use a string, or a `Decimal`, when you mean the decimal literal.

In [ ]:
from decimal import Decimal

print(Fraction(0.1))              # 3602879701896397/36028797018963968
print(Fraction("0.1"))            # 1/10
print(Fraction(Decimal("0.1")))   # 1/10 — Decimals are exact base-10, so this is fine
print(Fraction(0.25))             # 1/4 — exact because 0.25 is a power of two

## `limit_denominator` — find the nicest nearby fraction

`limit_denominator(max_den)` returns the closest fraction whose denominator is at most `max_den` (default 1 000 000). It's the cure for the float problem above, and a handy way to find simple approximations.

In [ ]:
import math

print(Fraction(0.1).limit_denominator())          # 1/10
print(Fraction(1 / 3).limit_denominator())        # 1/3
print(Fraction(math.pi).limit_denominator(10))    # 22/7
print(Fraction(math.pi).limit_denominator(1000))  # 355/113

**Tip:** this is great for human-friendly output — e.g. converting a measured `0.3125` inch into `5/16`, or showing a probability as `1/6` instead of `0.16666666666666666`.

In [ ]:
print(Fraction(0.3125).limit_denominator(64), "inch")
print(Fraction(1 / 6).limit_denominator(100))

## Exact arithmetic

All the usual operators work and stay exact:

In [ ]:
a, b = Fraction(1, 3), Fraction(1, 6)

print(a + b)          # 1/2
print(a - b)          # 1/6
print(a * 3)          # 1   — mixes with int
print(a / b)          # 2
print(a ** 2, a ** -1)  # 1/9 3
print(Fraction(7, 2) // 1, Fraction(7, 2) % 1)   # 3 1/2

Compare with `float`, where errors accumulate:

In [ ]:
harmonic = sum(Fraction(1, n) for n in range(1, 11))
print(harmonic)                        # exact: 7381/2520
print(sum(1 / n for n in range(1, 11)))  # float approximation

print(sum([Fraction(1, 10)] * 10) == 1)   # True — exactly 1

### Mixing with other number types

| Operation | Result |
|---|---|
| `Fraction` op `int` | `Fraction` (exact) |
| `Fraction` op `float` | `float` (exactness is lost!) |
| `Fraction` op `Decimal` | `TypeError` |
| `Fraction ** Fraction` with a non-integer exponent | `float` |

In [ ]:
f = Fraction(1, 3)
print(repr(f + 1))       # Fraction(4, 3)
print(repr(f + 0.5))     # 0.8333333333333333 — a float

try:
    f + Decimal("0.5")
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print(Fraction(2) ** Fraction(1, 2))   # sqrt(2) can't be rational -> float

⚠️ **Gotcha:** one stray `float` in a calculation silently turns the whole result into a `float`. Keep all inputs as `int`, `str` or `Fraction`.

## Inspecting and converting

`numerator` / `denominator` give you the parts; `as_integer_ratio()` returns them as a tuple. `int`, `float` and `Decimal` all have `as_integer_ratio()` too, which is handy for writing generic code.

In [ ]:
x = Fraction(-7, 4)
print(x.numerator, x.denominator)      # -7 4
print(x.as_integer_ratio())            # (-7, 4)

print((0.75).as_integer_ratio(), (5).as_integer_ratio(), Decimal("2.5").as_integer_ratio())

Converting out:

In [ ]:
x = Fraction(7, 2)
print(float(x), int(x))                # 3.5 3 — int() truncates toward zero
print(round(x), round(Fraction(5, 2))) # 4 2   — banker's rounding, like Math.Round
print(round(Fraction(22, 7), 2))       # 157/50 — rounding to n digits stays a Fraction
print(math.floor(Fraction(-7, 2)))     # -4

print(Fraction(6, 3).is_integer(), Fraction(5, 2).is_integer())   # 3.12+

## Comparing and formatting

Fractions compare correctly with every numeric type — against a `float` they compare by **exact value**:

In [ ]:
print(Fraction(1, 2) == 0.5)       # True — 0.5 is exact in binary
print(Fraction(1, 3) == 1 / 3)     # False — 1/3 as a float isn't exactly a third
print(sorted([Fraction(1, 3), 0.3, Fraction(2, 7)]))

`str()` gives `"n/d"`; since 3.12, fractions also accept float-style format specs like `.3f`:

In [ ]:
ratio = Fraction(22, 7)
print(str(ratio), repr(ratio))
print(f"{ratio:.3f}  {Fraction(1, 3):.2%}")

## `Fraction` vs `Decimal` vs `float`

| | `float` | `Decimal` | `Fraction` |
|---|---|---|---|
| Exact for | binary fractions (0.5, 0.25) | base-10 values (0.1, 19.99) | **any** rational (1/3, 2/7) |
| Speed | fastest (hardware) | slower | slowest (bignum + GCD each step) |
| Size of result | fixed | bounded by precision | can grow without limit |
| Typical use | science, graphics, ML | money, accounting | exact maths, ratios, probabilities, tests |

**Tip:** fractions are excellent as **test oracles**: compute the exact answer with `Fraction`, then check that your fast `float` code is close to it.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Exact rational | custom struct over `BigInteger` | `Fraction(n, d)` |
| From text | hand-written parser | `Fraction("3/4")`, `Fraction("0.75")` |
| From a double | — | `Fraction(x).limit_denominator()` |
| Parts | `.Num`, `.Den` (your fields) | `.numerator`, `.denominator`, `.as_integer_ratio()` |
| Simplest approximation | — | `limit_denominator(max_den)` |
| To double | `(double)num / (double)den` | `float(f)` |
| Format | — | `str(f)` → `"3/4"`, `f"{f:.3f}"` |

## Exercises

1. A recipe needs `3/4` cup of flour per batch. How much flour do you need for `2 1/2` batches? Print the result as a fraction and as a mixed number (e.g. `1 7/8`).
2. Compute the probability of rolling a total of 7 with two dice as an exact `Fraction` (hint: count outcomes with a comprehension).
3. Convert the float `0.142857142857` into the simplest fraction with a denominator ≤ 100.
4. Show that `sum(Fraction(1, 2**k) for k in range(1, 21))` is exactly `1 - 1/2**20`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from fractions import Fraction

# 1 — divmod gives the whole part and the remainder
flour = Fraction(3, 4) * Fraction("5/2")
whole, rest = divmod(flour, 1)
print(flour, f"= {whole} {rest}")          # 15/8 = 1 7/8

# 2
outcomes = [(a, b) for a in range(1, 7) for b in range(1, 7)]
p = Fraction(sum(1 for a, b in outcomes if a + b == 7), len(outcomes))
print(p)                                   # 1/6

# 3
print(Fraction(0.142857142857).limit_denominator(100))   # 1/7

# 4
total = sum(Fraction(1, 2**k) for k in range(1, 21))
print(total, total == 1 - Fraction(1, 2**20))            # 1048575/1048576 True
```

</details>